In [2]:
from pathlib import Path
import great_expectations as gx

PROJECT_ROOT = Path(r"C:\Users\Ahlam\Qafza_Training")
GX_ROOT = PROJECT_ROOT / "gx"

print("Project root:", PROJECT_ROOT)
print("GX root:", GX_ROOT)
print("GX config exists:", (GX_ROOT / "great_expectations.yml").exists())

context = gx.get_context(
    mode="file",
    context_root_dir=GX_ROOT,
)

print(type(context).__name__)
print(context.root_directory)

Project root: C:\Users\Ahlam\Qafza_Training
GX root: C:\Users\Ahlam\Qafza_Training\gx
GX config exists: True
FileDataContext
C:\Users\Ahlam\Qafza_Training\gx


In [3]:
data_source = context.data_sources.get("olist_artifacts")
print(data_source)

base_directory: ..\data\artifacts
id: 22fcddcf-1080-425d-94e4-899460213f92
name: olist_artifacts
type: pandas_filesystem



In [7]:
context.data_sources.delete(name="olist_artifacts")

data_source = context.data_sources.add_pandas_filesystem(
    name="olist_artifacts",
    base_directory=Path(r"C:\Users\Ahlam\Qafza_Training\data\artifacts"),
)

print(data_source)
print("\nResolved base directory:")
print(data_source.base_directory.resolve())

base_directory: C:\Users\Ahlam\Qafza_Training\data\artifacts
id: 862dc565-be2c-47d4-8768-a4a05d6fdb1a
name: olist_artifacts
type: pandas_filesystem


Resolved base directory:
C:\Users\Ahlam\Qafza_Training\data\artifacts


In [6]:
print("Context root:")
print(context.root_directory)

print("\nData source base directory:")
print(data_source.base_directory)

print("\nData source context root:")
print(getattr(data_source, "data_context_root_directory", "<not found>"))

print("\nResolved base directory:")
print(data_source.base_directory.resolve())

print("\nExpected data directory exists:")
print(Path(r"C:\Users\Ahlam\Qafza_Training\data\artifacts").exists())

Context root:
C:\Users\Ahlam\Qafza_Training\gx

Data source base directory:
..\data\artifacts

Data source context root:
None

Resolved base directory:
C:\Users\data\artifacts

Expected data directory exists:
True


In [8]:
asset = data_source.add_parquet_asset(
    name="labeled_orders",
    glob_directive="02_labeled_orders1.parquet",
)

print(type(asset).__name__)
print(asset)

ParquetAsset
connect_options:
  glob_directive: 02_labeled_orders1.parquet
id: f90850ca-7275-4860-88d3-00c6d28a27fc
name: labeled_orders
type: parquet



In [9]:
import inspect

print(inspect.signature(asset.add_batch_definition_path))

(name: 'str', path: 'PathStr') -> 'BatchDefinition'


In [10]:
batch_definition = asset.add_batch_definition_path(
    name="labeled_orders_batch",
    path="02_labeled_orders1.parquet",
)

print(type(batch_definition).__name__)
print(batch_definition)

BatchDefinition
id=UUID('3ed43dc6-99ed-4e5a-8af1-56c50bd4e442') name='labeled_orders_batch' partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True)


In [11]:
batch = batch_definition.get_batch()

print("Batch type:")
print(type(batch).__name__)

print("\nBatch data type:")
print(type(batch.data).__name__)

print("\nBatch data:")
print(batch.data)

Batch type:
Batch

Batch data type:
PandasBatchData

Batch data:


In [12]:
batch.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15

In [13]:
suite = gx.ExpectationSuite(
    name="olist_labeled_orders_quality"
)

suite = context.suites.add(suite)

print(type(suite).__name__)
print(suite.name)
print(len(suite.expectations))

ExpectationSuite
olist_labeled_orders_quality
0


In [14]:
expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
    column="is_delayed"
)

validation_result = batch.validate(expectation)

print("Expectation:")
print(expectation)

print("\nValidation success:")
print(validation_result.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Expectation:
id=None meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=True rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='is_delayed' mostly=1 row_condition=None condition_parser=None

Validation success:
True


In [15]:
suite.add_expectation(expectation)
suite.save()

print("Number of expectations:", len(suite.expectations))

Number of expectations: 1


In [16]:
for exp in suite.expectations:
    print(type(exp).__name__)
    print(exp.column)

ExpectColumnValuesToNotBeNull
is_delayed


In [17]:
import inspect

print(
    inspect.signature(
        gx.expectations.ExpectColumnValuesToBeInSet
    )
)

(*, id: Optional[str] = None, meta: Optional[dict] = None, notes: Union[str, List[str], NoneType] = None, result_format: Union[great_expectations.core.result_format.ResultFormat, dict] = <ResultFormat.BASIC: 'BASIC'>, description: Optional[str] = None, catch_exceptions: bool = True, rendered_content: Optional[List[great_expectations.render.components.RenderedAtomicContent]] = None, severity: great_expectations.expectations.metadata_types.FailureSeverity = <FailureSeverity.CRITICAL: 'critical'>, windows: Optional[List[great_expectations.expectations.window.Window]] = None, batch_id: Optional[str] = None, column: pydantic.v1.schema.StrictStr, mostly: Union[pydantic.v1.types.ConstrainedFloatValue, dict] = 1, row_condition: Union[str, great_expectations.expectations.row_conditions.ComparisonCondition, great_expectations.expectations.row_conditions.NullityCondition, great_expectations.expectations.row_conditions.AndCondition, great_expectations.expectations.row_conditions.OrCondition, great

In [18]:
expectation = gx.expectations.ExpectColumnValuesToBeInSet(
    column="is_delayed",
    value_set=[0, 1],
)

validation_result = batch.validate(expectation)

print("Validation success:")
print(validation_result.success)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Validation success:
True


In [19]:
suite.add_expectation(expectation)
suite.save()

print("Number of expectations:", len(suite.expectations))

Number of expectations: 2


In [20]:
for exp in suite.expectations:
    print(type(exp).__name__, "->", exp.column)

ExpectColumnValuesToNotBeNull -> is_delayed
ExpectColumnValuesToBeInSet -> is_delayed


In [21]:
expectation = gx.expectations.ExpectColumnValuesToBeInSet(
    column="order_status",
    value_set=["delivered"],
)

validation_result = batch.validate(expectation)

print("Validation success:")
print(validation_result.success)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Validation success:
True


In [22]:
import pandas as pd
import great_expectations as gx
from pathlib import Path


# --------------------------------------------------
# 1. Load the same dataset used by the GX Batch
# --------------------------------------------------

data_path = PROJECT_ROOT / "data" / "artifacts" / "02_labeled_orders1.parquet"

df = pd.read_parquet(data_path)


# --------------------------------------------------
# 2. Classify columns by data type
# --------------------------------------------------

numeric_columns = df.select_dtypes(include="number").columns.tolist()

datetime_columns = df.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()

categorical_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

other_columns = [
    column
    for column in df.columns
    if column not in (
        numeric_columns
        + datetime_columns
        + categorical_columns
    )
]


print("Dataset shape:", df.shape)

print("\nNumeric columns:", len(numeric_columns))
print(numeric_columns)

print("\nDatetime columns:", len(datetime_columns))
print(datetime_columns)

print("\nCategorical / text columns:", len(categorical_columns))
print(categorical_columns)

print("\nOther columns:", len(other_columns))
print(other_columns)


# --------------------------------------------------
# 3. Missing-value summary
# --------------------------------------------------

missing_counts = df.isna().sum()
missing_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("\nColumns containing missing values:")
if missing_columns.empty:
    print("None")
else:
    print(missing_columns)


# --------------------------------------------------
# 4. Helper: avoid duplicate expectations
# --------------------------------------------------

def has_expectation(expectation_type, column):
    return any(
        type(exp).__name__ == expectation_type
        and getattr(exp, "column", None) == column
        for exp in suite.expectations
    )


added_expectations = []


# --------------------------------------------------
# 5. Add Not-Null expectations for columns that
#    currently have zero missing values
# --------------------------------------------------

for column in df.columns:

    if missing_counts[column] == 0:

        if not has_expectation(
            "ExpectColumnValuesToNotBeNull",
            column,
        ):
            expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
                column=column
            )

            suite.add_expectation(expectation)
            added_expectations.append(
                f"NotNull -> {column}"
            )


# --------------------------------------------------
# 6. Target-specific rule
# --------------------------------------------------

if not has_expectation(
    "ExpectColumnValuesToBeInSet",
    "is_delayed",
):

    expectation = gx.expectations.ExpectColumnValuesToBeInSet(
        column="is_delayed",
        value_set=[0, 1],
    )

    suite.add_expectation(expectation)

    added_expectations.append(
        "InSet -> is_delayed = {0, 1}"
    )


# --------------------------------------------------
# 7. Dataset-specific categorical rule
# --------------------------------------------------

if not has_expectation(
    "ExpectColumnValuesToBeInSet",
    "order_status",
):

    expectation = gx.expectations.ExpectColumnValuesToBeInSet(
        column="order_status",
        value_set=["delivered"],
    )

    suite.add_expectation(expectation)

    added_expectations.append(
        'InSet -> order_status = {"delivered"}'
    )


# --------------------------------------------------
# 8. Label-construction date columns
# --------------------------------------------------

label_date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in label_date_columns:

    if not has_expectation(
        "ExpectColumnValuesToNotBeNull",
        column,
    ):

        expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
            column=column
        )

        suite.add_expectation(expectation)
        added_expectations.append(
            f"Label-date NotNull -> {column}"
        )


# --------------------------------------------------
# 9. Validate the whole Suite once
# --------------------------------------------------

print("\nExpectations added in this run:")

if added_expectations:
    for item in added_expectations:
        print("-", item)
else:
    print("None")


print("\nTotal expectations in suite before validation:")
print(len(suite.expectations))


validation_result = batch.validate(suite)

print("\nSuite validation success:")
print(validation_result.success)


# --------------------------------------------------
# 10. Save only after successful validation
# --------------------------------------------------

if validation_result.success:
    suite.save()
    print("\nSuite saved successfully.")
else:
    print("\nSuite was NOT saved because validation failed.")

Dataset shape: (96470, 42)

Numeric columns: 31
['customer_lat', 'customer_lng', 'customer_geo_points', 'seller_lat', 'seller_lng', 'seller_geo_points', 'num_sellers', 'num_seller_states', 'num_seller_cities', 'total_price', 'total_freight', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_payment', 'num_payment_records', 'num_payment_types', 'avg_review_score', 'num_reviews', 'avg_product_weight_g', 'total_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'total_product_volume_cm3', 'avg_product_photos_qty', 'avg_product_name_length', 'avg_product_description_length', 'seller_customer_distance_km', 'customer_zip_code_prefix', 'is_delayed']

Datetime columns: 5
['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Categorical / text columns: 6
['order_id', 'customer_id', 'order_status', 'customer_unique_id', 'customer_city', 'customer

Calculating Metrics:   0%|          | 0/123 [00:00<?, ?it/s]


Suite validation success:
True

Suite saved successfully.


In [23]:
print("Total expectations:", len(suite.expectations))

for i, exp in enumerate(suite.expectations, start=1):
    print(
        f"{i}. {type(exp).__name__}"
        f" | column={getattr(exp, 'column', None)}"
    )

Total expectations: 24
1. ExpectColumnValuesToNotBeNull | column=is_delayed
2. ExpectColumnValuesToBeInSet | column=is_delayed
3. ExpectColumnValuesToNotBeNull | column=order_id
4. ExpectColumnValuesToNotBeNull | column=customer_id
5. ExpectColumnValuesToNotBeNull | column=order_status
6. ExpectColumnValuesToNotBeNull | column=order_purchase_timestamp
7. ExpectColumnValuesToNotBeNull | column=order_delivered_customer_date
8. ExpectColumnValuesToNotBeNull | column=order_estimated_delivery_date
9. ExpectColumnValuesToNotBeNull | column=seller_geo_points
10. ExpectColumnValuesToNotBeNull | column=num_sellers
11. ExpectColumnValuesToNotBeNull | column=num_seller_states
12. ExpectColumnValuesToNotBeNull | column=num_seller_cities
13. ExpectColumnValuesToNotBeNull | column=total_price
14. ExpectColumnValuesToNotBeNull | column=total_freight
15. ExpectColumnValuesToNotBeNull | column=num_items
16. ExpectColumnValuesToNotBeNull | column=num_unique_products
17. ExpectColumnValuesToNotBeNull | c

In [24]:
saved_suite = context.suites.get("olist_labeled_orders_quality")

print("\nSaved suite expectations:", len(saved_suite.expectations))


Saved suite expectations: 24


In [25]:
print(context.validation_definitions.all())

[]


In [26]:
validation_definition = gx.ValidationDefinition(
    name="olist_labeled_orders_validation",
    data=batch_definition,
    suite=saved_suite,
)

validation_definition = context.validation_definitions.add(
    validation_definition
)

print(type(validation_definition).__name__)
print(validation_definition)

ValidationDefinition
name='olist_labeled_orders_validation' data=BatchDefinition(id=UUID('3ed43dc6-99ed-4e5a-8af1-56c50bd4e442'), name='labeled_orders_batch', partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True)) suite={
  "name": "olist_labeled_orders_quality",
  "id": "080d9899-b671-4dd4-815c-91218fff9dc4",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "is_delayed"
      },
      "meta": {},
      "id": "07ddad53-f25d-47d4-a46e-e2589b14613e",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_in_set",
      "kwargs": {
        "column": "is_delayed",
        "value_set": [
          0,
          1
        ]
      },
      "meta": {},
      "id": "4a41e754-b310-4e9e-8590-600ac0f7dd03",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "o

In [27]:
saved_validation = context.validation_definitions.get(
    "olist_labeled_orders_validation"
)

print("Validation Definition:")
print(saved_validation)

validation_results = saved_validation.run()

print("\nResult type:")
print(type(validation_results).__name__)

print("\nOverall success:")
print(validation_results.success)

print("\nTotal expectations evaluated:")
print(len(validation_results.results))

successful = sum(
    result.success
    for result in validation_results.results
)

failed = len(validation_results.results) - successful

print("\nSuccessful expectations:", successful)
print("Failed expectations:", failed)

Validation Definition:
name='olist_labeled_orders_validation' data=BatchDefinition(id=UUID('3ed43dc6-99ed-4e5a-8af1-56c50bd4e442'), name='labeled_orders_batch', partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True)) suite={
  "name": "olist_labeled_orders_quality",
  "id": "080d9899-b671-4dd4-815c-91218fff9dc4",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "is_delayed"
      },
      "meta": {},
      "id": "07ddad53-f25d-47d4-a46e-e2589b14613e",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_in_set",
      "kwargs": {
        "column": "is_delayed",
        "value_set": [
          0,
          1
        ]
      },
      "meta": {},
      "id": "4a41e754-b310-4e9e-8590-600ac0f7dd03",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": 

Calculating Metrics:   0%|          | 0/123 [00:00<?, ?it/s]


Result type:
ExpectationSuiteValidationResult

Overall success:
True

Total expectations evaluated:
24

Successful expectations: 24
Failed expectations: 0


In [28]:
import inspect

print(inspect.signature(gx.Checkpoint))

(*, name: str, validation_definitions: List[great_expectations.core.validation_definition.ValidationDefinition], actions: List[great_expectations.checkpoint.actions.ValidationAction] = None, result_format: Union[great_expectations.core.result_format.ResultFormat, dict, Literal['BOOLEAN_ONLY', 'BASIC', 'SUMMARY', 'COMPLETE']] = <ResultFormat.SUMMARY: 'SUMMARY'>, id: Optional[str] = None) -> None


In [29]:
print(context.checkpoints.all())

[]


In [30]:
validation_definition = context.validation_definitions.get(
    "olist_labeled_orders_validation"
)

checkpoint = gx.Checkpoint(
    name="olist_labeled_orders_checkpoint",
    validation_definitions=[validation_definition],
)

checkpoint = context.checkpoints.add(checkpoint)

print(type(checkpoint).__name__)
print(checkpoint)

Checkpoint
name='olist_labeled_orders_checkpoint' validation_definitions=[ValidationDefinition(name='olist_labeled_orders_validation', data=BatchDefinition(id=UUID('3ed43dc6-99ed-4e5a-8af1-56c50bd4e442'), name='labeled_orders_batch', partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True)), suite={
  "name": "olist_labeled_orders_quality",
  "id": "080d9899-b671-4dd4-815c-91218fff9dc4",
  "expectations": [
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "is_delayed"
      },
      "meta": {},
      "id": "07ddad53-f25d-47d4-a46e-e2589b14613e",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_in_set",
      "kwargs": {
        "column": "is_delayed",
        "value_set": [
          0,
          1
        ]
      },
      "meta": {},
      "id": "4a41e754-b310-4e9e-8590-600ac0f7dd03",
      "severity": "critical"
    },
    {
      "type": "

In [31]:
checkpoint = context.checkpoints.get(
    "olist_labeled_orders_checkpoint"
)

checkpoint_result = checkpoint.run()

print("Result type:")
print(type(checkpoint_result).__name__)

print("\nOverall success:")
print(checkpoint_result.success)

print("\nNumber of validation results:")
print(len(checkpoint_result.run_results))

Calculating Metrics:   0%|          | 0/123 [00:00<?, ?it/s]

Result type:
CheckpointResult

Overall success:
True

Number of validation results:
1


In [33]:
result = next(iter(checkpoint_result.run_results.values()))

print("Result type:", type(result).__name__)
print("Overall success:", result.success)
print("Total expectations:", len(result.results))

successful = sum(
    item.success
    for item in result.results
)

print("Successful:", successful)
print("Failed:", len(result.results) - successful)

Result type: ExpectationSuiteValidationResult
Overall success: True
Total expectations: 24
Successful: 24
Failed: 0


In [1]:
import inspect
import great_expectations as gx

context = gx.get_context()

print("Data sources methods:")
print([name for name in dir(context.data_sources) if not name.startswith("_")])

print("\nAdd pandas signature:")
print(inspect.signature(context.data_sources.add_pandas))

Data sources methods:
['add_alloy', 'add_aurora', 'add_bigquery', 'add_citus', 'add_databricks_sql', 'add_fabric', 'add_fabric_powerbi', 'add_neon', 'add_or_update_alloy', 'add_or_update_aurora', 'add_or_update_bigquery', 'add_or_update_citus', 'add_or_update_databricks_sql', 'add_or_update_fabric', 'add_or_update_fabric_powerbi', 'add_or_update_neon', 'add_or_update_pandas', 'add_or_update_pandas_abs', 'add_or_update_pandas_dbfs', 'add_or_update_pandas_filesystem', 'add_or_update_pandas_gcs', 'add_or_update_pandas_s3', 'add_or_update_postgres', 'add_or_update_redshift', 'add_or_update_snowflake', 'add_or_update_spark', 'add_or_update_spark_abs', 'add_or_update_spark_dbfs', 'add_or_update_spark_filesystem', 'add_or_update_spark_gcs', 'add_or_update_spark_s3', 'add_or_update_sql', 'add_or_update_sql_server', 'add_or_update_sqlite', 'add_pandas', 'add_pandas_abs', 'add_pandas_dbfs', 'add_pandas_filesystem', 'add_pandas_gcs', 'add_pandas_s3', 'add_postgres', 'add_redshift', 'add_snowflake

In [2]:
import inspect

context = gx.get_context()

pandas_source = context.data_sources.pandas_default

print("Pandas datasource type:")
print(type(pandas_source))

print("\nDatasource methods:")
print([
    name for name in dir(pandas_source)
    if "asset" in name.lower() or "batch" in name.lower()
])

print("\nadd_dataframe_asset signature:")
print(inspect.signature(pandas_source.add_dataframe_asset))

Pandas datasource type:
<class 'great_expectations.datasource.fluent.pandas_datasource.PandasDatasource'>

Datasource methods:
['_add_asset', '_get_batch', '_load_asset_subtype', '_rebuild_asset_data_connectors', '_update_asset_forward_refs', '_update_batch_definitions', '_validate_asset_name', 'add_batch_definition', 'add_clipboard_asset', 'add_csv_asset', 'add_dataframe_asset', 'add_excel_asset', 'add_feather_asset', 'add_fwf_asset', 'add_gbq_asset', 'add_hdf_asset', 'add_html_asset', 'add_json_asset', 'add_orc_asset', 'add_parquet_asset', 'add_pickle_asset', 'add_sas_asset', 'add_spss_asset', 'add_sql_asset', 'add_sql_query_asset', 'add_sql_table_asset', 'add_stata_asset', 'add_table_asset', 'add_xml_asset', 'asset_types', 'assets', 'delete_asset', 'delete_batch_definition', 'get_asset', 'get_asset_names', 'get_assets_as_dict', 'get_batch', 'get_batch_identifiers_list']

add_dataframe_asset signature:
(name: 'str', batch_metadata: 'Optional[BatchMetadata]' = None) -> 'DataFrameAsset

In [4]:
test_asset = pandas_source.add_dataframe_asset(
    name="test_inference_asset"
)

print("Asset type:")
print(type(test_asset))

print("\nBatch-related methods:")
print([
    name
    for name in dir(test_asset)
    if "batch" in name.lower()
])

print("\nDatasource add_batch_definition:")
print(inspect.signature(pandas_source.add_batch_definition))

Asset type:
<class 'great_expectations.datasource.fluent.pandas_datasource.DataFrameAsset'>

Batch-related methods:
['_batch_parameters_are_valid', '_delete_batch_definition', '_get_batch_metadata_from_batch_request', '_sort_batch_data_list', '_validate_batch_request', 'add_batch_definition', 'add_batch_definition_whole_dataframe', 'batch_definitions', 'batch_metadata', 'build_batch_request', 'delete_batch_definition', 'ensure_batch_metadata_is_not_none', 'get_batch', 'get_batch_definition', 'get_batch_identifiers_list', 'get_batch_parameters_keys', 'sort_batch_identifiers_list', 'sort_batches', 'sort_legacy_batch_definitions', 'update_batch_definition_field_set']

Datasource add_batch_definition:
(batch_definition: 'BatchDefinition[PartitionerT]') -> 'BatchDefinition[PartitionerT]'


In [5]:
import inspect

print("DataFrameAsset methods:")
print([
    name
    for name in dir(pandas_source.get_asset("test_inference_asset"))
    if "batch" in name.lower()
])

DataFrameAsset methods:
['_batch_parameters_are_valid', '_delete_batch_definition', '_get_batch_metadata_from_batch_request', '_sort_batch_data_list', '_validate_batch_request', 'add_batch_definition', 'add_batch_definition_whole_dataframe', 'batch_definitions', 'batch_metadata', 'build_batch_request', 'delete_batch_definition', 'ensure_batch_metadata_is_not_none', 'get_batch', 'get_batch_definition', 'get_batch_identifiers_list', 'get_batch_parameters_keys', 'sort_batch_identifiers_list', 'sort_batches', 'sort_legacy_batch_definitions', 'update_batch_definition_field_set']


In [6]:
import inspect

print(
    inspect.signature(
        test_asset.add_batch_definition_whole_dataframe
    )
)

(name: 'str') -> 'BatchDefinition'


In [7]:
import pandas as pd

data = pd.read_parquet(
    "data/artifacts/03_test1.parquet"
).drop(columns=["is_delayed"]).head(10)

print(data.shape)
print(data.columns.tolist())

(10, 41)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_lat', 'customer_lng', 'customer_geo_points', 'seller_lat', 'seller_lng', 'seller_geo_points', 'num_sellers', 'num_seller_states', 'num_seller_cities', 'total_price', 'total_freight', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_payment', 'num_payment_records', 'num_payment_types', 'avg_review_score', 'num_reviews', 'avg_product_weight_g', 'total_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'total_product_volume_cm3', 'avg_product_photos_qty', 'avg_product_name_length', 'avg_product_description_length', 'seller_customer_distance_km', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [8]:
batch_definition = test_asset.add_batch_definition_whole_dataframe(
    name="test_inference_batch"
)

print("Batch definition:")
print(batch_definition)

Batch definition:
id=UUID('19bb674b-9de8-4244-bf40-ad31b4d24f1a') name='test_inference_batch' partitioner=None


In [9]:
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": data}
)

print("Batch type:")
print(type(batch))

print("\nBatch data type:")
print(type(batch.data))

print("\nBatch data shape:")
print(batch.data.shape)

Batch type:
<class 'great_expectations.datasource.fluent.interfaces.Batch'>

Batch data type:
<class 'great_expectations.execution_engine.pandas_batch_data.PandasBatchData'>

Batch data shape:


AttributeError: 'PandasBatchData' object has no attribute 'shape'

In [10]:
print("Batch methods:")
print([
    name
    for name in dir(batch)
    if not name.startswith("_")
])

print("\nBatch data methods:")
print([
    name
    for name in dir(batch.data)
    if not name.startswith("_")
])

Batch methods:
['batch_definition', 'batch_markers', 'batch_request', 'batch_spec', 'columns', 'compute_metrics', 'data', 'data_asset', 'datasource', 'head', 'id', 'metadata', 'validate']

Batch data methods:
['dataframe', 'execution_engine', 'head']


In [11]:
print("DataFrame type:")
print(type(batch.data.dataframe))

print("\nDataFrame shape:")
print(batch.data.dataframe.shape)

print("\nColumns:")
print(batch.data.dataframe.columns.tolist())

DataFrame type:
<class 'pandas.core.frame.DataFrame'>

DataFrame shape:
(10, 41)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_lat', 'customer_lng', 'customer_geo_points', 'seller_lat', 'seller_lng', 'seller_geo_points', 'num_sellers', 'num_seller_states', 'num_seller_cities', 'total_price', 'total_freight', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_payment', 'num_payment_records', 'num_payment_types', 'avg_review_score', 'num_reviews', 'avg_product_weight_g', 'total_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'total_product_volume_cm3', 'avg_product_photos_qty', 'avg_product_name_length', 'avg_product_description_length', 'seller_customer_distance_km', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [12]:
print("Pandas assets:")
print(pandas_source.get_asset_names())

print("\nBatch definitions:")
print(test_asset.get_batch_identifiers_list())

Pandas assets:
{'test_inference_asset'}

Batch definitions:


TypeError: DataFrameAsset.get_batch_identifiers_list() missing 1 required positional argument: 'batch_request'

In [13]:
pandas_source.delete_asset("test_inference_asset")

print("Pandas assets after cleanup:")
print(pandas_source.get_asset_names())

Pandas assets after cleanup:
set()


In [14]:
import great_expectations as gx

context = gx.get_context()

inference_source = context.data_sources.add_pandas(
    name="olist_inference"
)

inference_asset = inference_source.add_dataframe_asset(
    name="inference_requests"
)

inference_batch_definition = (
    inference_asset.add_batch_definition_whole_dataframe(
        name="inference_batch"
    )
)

print("Data source:", inference_source.name)
print("Asset:", inference_asset.name)
print("Batch definition:", inference_batch_definition.name)

Data source: olist_inference
Asset: inference_requests
Batch definition: inference_batch


In [15]:
suite = gx.ExpectationSuite(
    name="olist_inference_input_quality"
)

required_columns = [
    "seller_customer_distance_km",
    "total_freight",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "customer_state",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

for column in required_columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnToExist(
            column=column
        )
    )

print("Suite name:", suite.name)
print("Number of expectations:", len(suite.expectations))

Suite name: olist_inference_input_quality
Number of expectations: 11


In [16]:
numeric_expectations = [
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="total_freight",
        min_value=0,
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="total_price",
        min_value=0,
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="num_items",
        min_value=0,
        strict_min=True,
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="num_unique_products",
        min_value=0,
        strict_min=True,
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="num_unique_sellers",
        min_value=0,
        strict_min=True,
    ),
]

for expectation in numeric_expectations:
    suite.add_expectation(expectation)

print("Number of expectations:", len(suite.expectations))

Number of expectations: 16


In [17]:
brazilian_states = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF",
    "ES", "GO", "MA", "MG", "MS", "MT", "PA",
    "PB", "PE", "PI", "PR", "RJ", "RN", "RO",
    "RR", "RS", "SC", "SE", "SP", "TO",
]

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="customer_state",
        value_set=brazilian_states,
    )
)

print("Number of expectations:", len(suite.expectations))

Number of expectations: 17


In [18]:
valid_data = data.copy()

valid_data.loc[valid_data.index[0], "customer_state"] = None

test_asset = inference_source.add_dataframe_asset(
    name="temporary_validation_test"
)

test_batch_definition = (
    test_asset.add_batch_definition_whole_dataframe(
        name="temporary_batch"
    )
)

test_batch = test_batch_definition.get_batch(
    batch_parameters={"dataframe": valid_data}
)

result = test_batch.validate(suite)

print("Validation success:", result.success)

for result_item in result.results:
    if not result_item.success:
        print(result_item.expectation_config)
        print(result_item.result)

Calculating Metrics:   0%|          | 0/45 [00:00<?, ?it/s]

Validation success: True


In [19]:
inference_source.delete_asset("temporary_validation_test")

print("Pandas assets:")
print(inference_source.get_asset_names())

Pandas assets:
{'inference_requests'}


In [20]:
date_expectations = [
    gx.expectations.ExpectColumnValuesToBeDateutilParseable(
        column="order_purchase_timestamp"
    ),
    gx.expectations.ExpectColumnValuesToBeDateutilParseable(
        column="order_estimated_delivery_date"
    ),
]

for expectation in date_expectations:
    suite.add_expectation(expectation)

print("Number of expectations:", len(suite.expectations))

Number of expectations: 19


In [21]:
train_data = pd.read_parquet(
    "data/artifacts/03_train1.parquet"
)

inference_columns = [
    "seller_customer_distance_km",
    "total_freight",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "customer_state",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

missing_rates = (
    train_data[inference_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print(missing_rates)

seller_customer_distance_km      0.005094
total_freight                    0.000000
total_price                      0.000000
num_items                        0.000000
num_unique_products              0.000000
num_unique_sellers               0.000000
total_product_weight_g           0.000000
total_product_volume_cm3         0.000000
customer_state                   0.000000
order_purchase_timestamp         0.000000
order_estimated_delivery_date    0.000000
dtype: float64


In [22]:
numeric_input_columns = [
    "seller_customer_distance_km",
    "total_freight",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3",
]

for column in numeric_input_columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnProportionOfNonNullValuesToBeBetween(
            column=column,
            min_value=0.0,
            strict_min=True,
        )
    )

print("Number of expectations:", len(suite.expectations))


Number of expectations: 27


In [23]:
valid_data = (
    pd.read_parquet("data/artifacts/03_test1.parquet")
    .drop(columns=["is_delayed"])
    .head(10)
)

valid_batch = inference_batch_definition.get_batch(
    batch_parameters={"dataframe": valid_data}
)

valid_result = valid_batch.validate(suite)

print("Validation success:", valid_result.success)

failed = [
    result
    for result in valid_result.results
    if not result.success
]

print("Failed expectations:", len(failed))

Calculating Metrics:   0%|          | 0/75 [00:00<?, ?it/s]

Validation success: False
Failed expectations: 2


In [24]:
for result in valid_result.results:
    if not result.success:
        print("Expectation:")
        print(result.expectation_config)
        print("\nResult:")
        print(result.result)
        print("\n" + "=" * 80)

Expectation:
{
  "type": "expect_column_values_to_be_dateutil_parseable",
  "kwargs": {
    "column": "order_purchase_timestamp",
    "batch_id": "olist_inference-inference_requests"
  },
  "meta": {},
  "severity": "critical"
}

Result:
{}

Expectation:
{
  "type": "expect_column_values_to_be_dateutil_parseable",
  "kwargs": {
    "column": "order_estimated_delivery_date",
    "batch_id": "olist_inference-inference_requests"
  },
  "meta": {},
  "severity": "critical"
}

Result:
{}



In [25]:
import inspect

print(inspect.signature(suite.remove_expectation))


(expectation_configuration: 'Optional[ExpectationConfiguration]' = None, match_type: 'str' = 'domain', remove_multiple_matches: 'bool' = False, id: 'Optional[Union[str, uuid.UUID]]' = None) -> 'List[ExpectationConfiguration]'


In [26]:
from great_expectations.expectations.expectation_configuration import (
    ExpectationConfiguration,
)

for column in [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]:
    removed = suite.remove_expectation(
        expectation_configuration=ExpectationConfiguration(
            type="expect_column_values_to_be_dateutil_parseable",
            kwargs={"column": column},
        )
    )
    print(column, "removed:", len(removed))

print("Number of expectations:", len(suite.expectations))

order_purchase_timestamp removed: 1
order_estimated_delivery_date removed: 1
Number of expectations: 25


In [27]:
import inspect

print(
    inspect.signature(
        gx.expectations.ExpectColumnValuesToBeOfType
    )
)

(*, id: Optional[str] = None, meta: Optional[dict] = None, notes: Union[str, List[str], NoneType] = None, result_format: Union[great_expectations.core.result_format.ResultFormat, dict] = <ResultFormat.BASIC: 'BASIC'>, description: Optional[str] = None, catch_exceptions: bool = True, rendered_content: Optional[List[great_expectations.render.components.RenderedAtomicContent]] = None, severity: great_expectations.expectations.metadata_types.FailureSeverity = <FailureSeverity.CRITICAL: 'critical'>, windows: Optional[List[great_expectations.expectations.window.Window]] = None, batch_id: Optional[str] = None, column: pydantic.v1.schema.StrictStr, mostly: Union[pydantic.v1.types.ConstrainedFloatValue, dict] = 1, row_condition: Union[str, great_expectations.expectations.row_conditions.ComparisonCondition, great_expectations.expectations.row_conditions.NullityCondition, great_expectations.expectations.row_conditions.AndCondition, great_expectations.expectations.row_conditions.OrCondition, great

In [28]:
print("purchase:", valid_data["order_purchase_timestamp"].dtype)
print("estimated:", valid_data["order_estimated_delivery_date"].dtype)

purchase: datetime64[ns]
estimated: datetime64[ns]


In [29]:
test_suite = gx.ExpectationSuite(name="test_datetime_type")

test_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="order_purchase_timestamp",
        type_="datetime64[ns]",
    )
)

test_batch = inference_batch_definition.get_batch(
    batch_parameters={"dataframe": valid_data}
)

test_result = test_batch.validate(test_suite)

print("Validation success:", test_result.success)
print("Failed expectations:", [
    r["expectation_config"]["type"]
    for r in test_result.results
    if not r["success"]
])

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Validation success: True
Failed expectations: []


In [30]:
for column in [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeOfType(
            column=column,
            type_="datetime64[ns]",
        )
    )

print("Number of expectations:", len(suite.expectations))

Number of expectations: 27


In [31]:
valid_result = valid_batch.validate(suite)

print("Validation success:", valid_result.success)

failed = [
    r["expectation_config"]["type"]
    for r in valid_result.results
    if not r["success"]
]

print("Failed expectations:", failed)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation success: True
Failed expectations: []


In [32]:
invalid_data = valid_data.copy()
invalid_data.loc[invalid_data.index[0], "total_freight"] = -1

invalid_batch = inference_batch_definition.get_batch(
    batch_parameters={"dataframe": invalid_data}
)

invalid_result = invalid_batch.validate(suite)

print("Validation success:", invalid_result.success)

failed = [
    r["expectation_config"]["type"]
    for r in invalid_result.results
    if not r["success"]
]

print("Failed expectations:", failed)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation success: False
Failed expectations: ['expect_column_values_to_be_between']


In [33]:
invalid_state_data = valid_data.copy()
invalid_state_data.loc[invalid_state_data.index[0], "customer_state"] = "XX"

invalid_state_batch = inference_batch_definition.get_batch(
    batch_parameters={"dataframe": invalid_state_data}
)

invalid_state_result = invalid_state_batch.validate(suite)

print("Validation success:", invalid_state_result.success)

failed = [
    r["expectation_config"]["type"]
    for r in invalid_state_result.results
    if not r["success"]
]

print("Failed expectations:", failed)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation success: False
Failed expectations: ['expect_column_values_to_be_in_set']


In [34]:
all_missing_data = valid_data.copy()
all_missing_data["total_price"] = float("nan")

all_missing_batch = inference_batch_definition.get_batch(
    batch_parameters={"dataframe": all_missing_data}
)

all_missing_result = all_missing_batch.validate(suite)

print("Validation success:", all_missing_result.success)

failed = [
    r["expectation_config"]["type"]
    for r in all_missing_result.results
    if not r["success"]
]

print("Failed expectations:", failed)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation success: False
Failed expectations: ['expect_column_proportion_of_non_null_values_to_be_between']


In [35]:
context.suites.add(suite)

saved_suite = context.suites.get("olist_inference_input_quality")

print("Suite name:", saved_suite.name)
print("Number of expectations:", len(saved_suite.expectations))

Suite name: olist_inference_input_quality
Number of expectations: 27


In [38]:
import inspect

print(type(context))

print("\nContext attributes related to config/files:")
for name in dir(context):
    if any(word in name.lower() for word in ["config", "root", "file", "store"]):
        print(name)

<class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>

Context attributes related to config/files:
GLOBAL_CONFIG_PATHS
_ETC_CONF_FILE
_ROOT_CONF_DIR
_ROOT_CONF_FILE
_attach_fluent_config_datasources_and_build_data_connectors
_build_store_from_config
_config_provider
_config_variables
_datasource_store
_get_global_config_value
_get_metric_configuration_tuples
_init_config_provider
_init_datasource_store
_init_primary_stores
_init_project_config
_load_config_variables
_load_fluent_config
_load_site_builder_from_site_config
_project_config
_save_project_config
_scaffold_root_conf
_store_metrics
_stores
_update_config_variables
add_store
checkpoint_store
checkpoint_store_name
config
config_provider
config_variables
convert_to_file_context
datasource_store
delete_store
escape_all_config_variables
expectations_store
expectations_store_name
fluent_config
get_config
get_config_with_variables_substituted
get_or_create_data_context_config
list_active_

In [39]:
print("\nSuite store type:")
print(type(context.suites))

print("\nSuite store attributes:")
for name in dir(context.suites):
    if not name.startswith("_"):
        print(name)


Suite store type:
<class 'great_expectations.core.factory.suite_factory.SuiteFactory'>

Suite store attributes:
add
add_or_update
all
delete
get


In [36]:
from pathlib import Path

expectations_dir = Path("gx/expectations")

print("Expectation files:")
for path in expectations_dir.iterdir():
    print("-", path.name)

Expectation files:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'gx\\expectations'

In [37]:
print("GE root:", context.root_directory)

expectations_dir = Path(context.root_directory) / "expectations"

print("Expectation directory:", expectations_dir)
print("Exists:", expectations_dir.exists())

if expectations_dir.exists():
    print("\nExpectation files:")
    for path in expectations_dir.iterdir():
        print("-", path.name)

GE root: None


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [40]:
print("Expectations store:")
print(context.expectations_store)

print("\nExpectations store config:")
print(context.expectations_store.config)

Expectations store:

Expectations store config:


NotImplementedError: 

In [41]:
print("\nActive stores:")
print(context.list_active_stores())


Active stores:
[{'class_name': 'ExpectationsStore', 'store_backend': {'class_name': 'InMemoryStoreBackend'}, 'name': 'expectations_store'}, {'class_name': 'ValidationResultsStore', 'store_backend': {'class_name': 'InMemoryStoreBackend'}, 'name': 'validation_results_store'}, {'class_name': 'CheckpointStore', 'store_backend': {'class_name': 'InMemoryStoreBackend'}, 'name': 'checkpoint_store'}]


In [42]:
import great_expectations as gx
import inspect

print(inspect.signature(gx.get_context))

print("\nFileDataContext:")
print(gx.data_context.FileDataContext if hasattr(gx.data_context, "FileDataContext") else "Not found")

(project_config: 'DataContextConfig | Mapping | None' = None, context_root_dir: 'PathStr | None' = None, project_root_dir: 'PathStr | None' = None, runtime_environment: 'dict | None' = None, cloud_base_url: 'str | None' = None, cloud_access_token: 'str | None' = None, cloud_organization_id: 'str | None' = None, cloud_workspace_id: 'str | None' = None, cloud_mode: 'bool | None' = None, user_agent_str: 'str | None' = None, mode: 'ContextModes | None' = None) -> 'AbstractDataContext'

FileDataContext:
<class 'great_expectations.data_context.data_context.file_data_context.FileDataContext'>


In [43]:
file_context = gx.get_context(
    context_root_dir=r"C:\Users\Ahlam\Qafza_Training\gx"
)

print(type(file_context))
print(file_context.root_directory)
print(file_context.list_active_stores())

<class 'great_expectations.data_context.data_context.file_data_context.FileDataContext'>
C:\Users\Ahlam\Qafza_Training\gx
[{'class_name': 'ExpectationsStore', 'store_backend': {'class_name': 'TupleFilesystemStoreBackend', 'base_directory': 'expectations/'}, 'name': 'expectations_store'}, {'class_name': 'ValidationResultsStore', 'store_backend': {'class_name': 'TupleFilesystemStoreBackend', 'base_directory': 'uncommitted/validations/'}, 'name': 'validation_results_store'}, {'class_name': 'CheckpointStore', 'store_backend': {'class_name': 'TupleFilesystemStoreBackend', 'suppress_store_backend_id': True, 'base_directory': 'checkpoints/'}, 'name': 'checkpoint_store'}]


In [44]:
file_context.suites.add(suite)

saved_suite = file_context.suites.get(
    "olist_inference_input_quality"
)

print("Suite name:", saved_suite.name)
print("Number of expectations:", len(saved_suite.expectations))

Suite name: olist_inference_input_quality
Number of expectations: 27


In [45]:
from pathlib import Path

expectations_dir = Path(r"C:\Users\Ahlam\Qafza_Training\gx\expectations")

print("Exists:", expectations_dir.exists())

print("\nExpectation files:")
for path in expectations_dir.iterdir():
    print("-", path.name)

Exists: True

Expectation files:
- .ge_store_backend_id
- olist_inference_input_quality.json
- olist_labeled_orders_quality.json


In [48]:
print("File context datasources:")
print(file_context.data_sources.all())

File context datasources:
{'olist_artifacts': PandasFilesystemDatasource(type='pandas_filesystem', name='olist_artifacts', id=UUID('862dc565-be2c-47d4-8768-a4a05d6fdb1a'), assets=[ParquetAsset(name='labeled_orders', type='parquet', id=UUID('f90850ca-7275-4860-88d3-00c6d28a27fc'), order_by=[], batch_metadata={}, batch_definitions=[BatchDefinition[Union[FileNamePartitionerYearly, FileNamePartitionerMonthly, FileNamePartitionerDaily, FileNamePartitionerPath]](id='3ed43dc6-99ed-4e5a-8af1-56c50bd4e442', name='labeled_orders_batch', partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True))], connect_options={'glob_directive': '02_labeled_orders1.parquet'}, engine='auto', columns=None, storage_options=None, use_nullable_dtypes=None, dtype_backend=None, kwargs=None)], base_directory=WindowsPath('C:/Users/Ahlam/Qafza_Training/data/artifacts'), data_context_root_directory=None)}


In [49]:
inference_source_file = file_context.data_sources.add_pandas(
    name="olist_inference"
)

print("Datasource:", inference_source_file.name)

Datasource: olist_inference


In [50]:
inference_asset_file = inference_source_file.add_dataframe_asset(
    name="inference_requests"
)

inference_batch_definition_file = (
    inference_asset_file.add_batch_definition_whole_dataframe(
        name="inference_batch"
    )
)

print("Datasource:", inference_source_file.name)
print("Asset:", inference_asset_file.name)
print("Batch definition:", inference_batch_definition_file.name)

Datasource: olist_inference
Asset: inference_requests
Batch definition: inference_batch


In [51]:
valid_batch_file = inference_batch_definition_file.get_batch(
    batch_parameters={"dataframe": valid_data}
)

loaded_result = valid_batch_file.validate(loaded_suite)

print("Validation success:", loaded_result.success)
print("Number of expectations:", len(loaded_suite.expectations))

failed = [
    r["expectation_config"]["type"]
    for r in loaded_result.results
    if not r["success"]
]

print("Failed expectations:", failed)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation success: True
Number of expectations: 27
Failed expectations: []


In [53]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Ahlam\Qafza_Training")
validation_path = PROJECT_ROOT / "src" / "validation.py"

print("Exists:", validation_path.exists())
print("Path:", validation_path)

print("\n--- validation.py ---")
print(validation_path.read_text(encoding="utf-8"))

Exists: True
Path: C:\Users\Ahlam\Qafza_Training\src\validation.py

--- validation.py ---
from dataclasses import dataclass

import pandas as pd


REQUIRED_COLUMNS = [
    "seller_customer_distance_km",
    "total_freight",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "customer_state",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]


@dataclass
class ValidationResult:
    valid: bool
    errors: list[str]


def validate_input(data: pd.DataFrame) -> ValidationResult:
    errors = []

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in data.columns
    ]

    if missing_columns:
        errors.append(
            f"Missing required columns: {missing_columns}"
        )

    if errors:
        return ValidationResult(
            valid=False,
            errors=errors,
        )

    numeric_columns = [
   

In [54]:
fresh_context = gx.get_context(
    context_root_dir=r"C:\Users\Ahlam\Qafza_Training\gx"
)

print("Datasources in fresh context:")
print(fresh_context.data_sources.all())

Datasources in fresh context:
{'olist_artifacts': PandasFilesystemDatasource(type='pandas_filesystem', name='olist_artifacts', id=UUID('862dc565-be2c-47d4-8768-a4a05d6fdb1a'), assets=[ParquetAsset(name='labeled_orders', type='parquet', id=UUID('f90850ca-7275-4860-88d3-00c6d28a27fc'), order_by=[], batch_metadata={}, batch_definitions=[BatchDefinition[Union[FileNamePartitionerYearly, FileNamePartitionerMonthly, FileNamePartitionerDaily, FileNamePartitionerPath]](id='3ed43dc6-99ed-4e5a-8af1-56c50bd4e442', name='labeled_orders_batch', partitioner=FileNamePartitionerPath(regex=re.compile('02_labeled_orders1.parquet$'), param_names=(), sort_ascending=True))], connect_options={'glob_directive': '02_labeled_orders1.parquet'}, engine='auto', columns=None, storage_options=None, use_nullable_dtypes=None, dtype_backend=None, kwargs=None)], base_directory=WindowsPath('C:/Users/Ahlam/Qafza_Training/data/artifacts'), data_context_root_directory=None), 'olist_inference': PandasDatasource(type='pandas'

In [55]:
import great_expectations as gx

context = gx.get_context(
    context_root_dir=r"C:\Users\Ahlam\Qafza_Training\gx"
)

source = context.data_sources.get("olist_inference")

print(type(source))
print([name for name in dir(source) if "asset" in name.lower()])

<class 'great_expectations.datasource.fluent.pandas_datasource.PandasDatasource'>
['_add_asset', '_load_asset_subtype', '_rebuild_asset_data_connectors', '_update_asset_forward_refs', '_validate_asset_name', 'add_clipboard_asset', 'add_csv_asset', 'add_dataframe_asset', 'add_excel_asset', 'add_feather_asset', 'add_fwf_asset', 'add_gbq_asset', 'add_hdf_asset', 'add_html_asset', 'add_json_asset', 'add_orc_asset', 'add_parquet_asset', 'add_pickle_asset', 'add_sas_asset', 'add_spss_asset', 'add_sql_asset', 'add_sql_query_asset', 'add_sql_table_asset', 'add_stata_asset', 'add_table_asset', 'add_xml_asset', 'asset_types', 'assets', 'delete_asset', 'get_asset', 'get_asset_names', 'get_assets_as_dict']


In [56]:
asset = source.get_asset("inference_requests")

print(type(asset))
print([name for name in dir(asset) if "batch" in name.lower()])

<class 'great_expectations.datasource.fluent.pandas_datasource.DataFrameAsset'>
['_batch_parameters_are_valid', '_delete_batch_definition', '_get_batch_metadata_from_batch_request', '_sort_batch_data_list', '_validate_batch_request', 'add_batch_definition', 'add_batch_definition_whole_dataframe', 'batch_definitions', 'batch_metadata', 'build_batch_request', 'delete_batch_definition', 'ensure_batch_metadata_is_not_none', 'get_batch', 'get_batch_definition', 'get_batch_identifiers_list', 'get_batch_parameters_keys', 'sort_batch_identifiers_list', 'sort_batches', 'sort_legacy_batch_definitions', 'update_batch_definition_field_set']


In [57]:
batch_definition = asset.get_batch_definition("inference_batch")

print(type(batch_definition))
print(batch_definition.name)

<class 'great_expectations.core.batch_definition.BatchDefinition'>
inference_batch


In [60]:
import os
from pathlib import Path

print("Current directory:")
print(os.getcwd())

print("\nvalidation.py exists:")
print(Path(r"C:\Users\Ahlam\Qafza_Training\src\validation.py").exists())

Current directory:
C:\Users\Ahlam

validation.py exists:
True


In [61]:
import sys

project_root = r"C:\Users\Ahlam\Qafza_Training"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[0])

C:\Users\Ahlam\Qafza_Training


In [62]:
import pandas as pd
from src.validation import validate_with_great_expectations

test_data = pd.read_parquet(
    r"C:\Users\Ahlam\Qafza_Training\data\artifacts\03_test1.parquet"
).drop(columns=["is_delayed"]).head(10)

errors = validate_with_great_expectations(test_data)

print("Errors:", errors)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Errors: []


In [63]:
invalid_data = test_data.copy()
invalid_data.loc[invalid_data.index[0], "total_freight"] = -1

errors = validate_with_great_expectations(invalid_data)

print("Errors:", errors)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Errors: ['Great Expectations validation failed: expect_column_values_to_be_between']


In [64]:
from src.validation import validate_input

result = validate_input(test_data)

print("Valid:", result.valid)
print("Errors:", result.errors)

Valid: True
Errors: []


In [69]:
invalid_data = test_data.copy()
invalid_data.loc[invalid_data.index[0], "total_freight"] = -1

result = validate_input(invalid_data)

print("Valid:", result.valid)
print("Errors:", result.errors)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Valid: False
Errors: ['Great Expectations validation failed: expect_column_values_to_be_between', 'total_freight cannot be negative.']


In [67]:
import importlib
import src.validation

importlib.reload(src.validation)

validate_input = src.validation.validate_input

In [68]:
invalid_state_data = test_data.copy()
invalid_state_data.loc[invalid_state_data.index[0], "customer_state"] = "XX"

result = validate_input(invalid_state_data)

print("Valid:", result.valid)
print("Errors:", result.errors)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Valid: False
Errors: ['Great Expectations validation failed: expect_column_values_to_be_in_set']


In [70]:
missing_data = test_data.copy()
missing_data["total_price"] = float("nan")

result = validate_input(missing_data)

print("Valid:", result.valid)
print("Errors:", result.errors)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Valid: False
Errors: ['Great Expectations validation failed: expect_column_proportion_of_non_null_values_to_be_between', 'total_price contains only missing values.']


In [71]:
from src.pipeline import run_inference

predictions, probabilities = run_inference(test_data)

print("Predictions:", len(predictions))
print("Probabilities:", len(probabilities))
print("First predictions:", predictions[:5])
print("First probabilities:", probabilities[:5])

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Predictions: 10
Probabilities: 10
First predictions: [0 0 0 0 0]
First probabilities: [0.44298556 0.44993345 0.09949204 0.28897331 0.427422  ]


In [75]:
%cd C:\Users\Ahlam\Qafza_Training

C:\Users\Ahlam\Qafza_Training


In [76]:
from pathlib import Path

path = Path("app/main.py")

print(path.read_text(encoding="utf-8"))


import time
import logging
import pandas as pd
from fastapi import FastAPI, HTTPException

from app.schemas import (
    BatchPredictionRequest,
    BatchPredictionResponse,
    HealthResponse,
    ModelInfoResponse,
    OrderRequest,
    PredictionResponse,
)
from src.config import load_config
from src.logging_config import configure_logging
from src.pipeline import run_inference


config = load_config()

configure_logging(
    log_directory=config["logging"]["directory"],
    log_filename=config["logging"]["filename"],
    log_level=config["logging"]["level"],
)

logger = logging.getLogger(__name__)

app = FastAPI(
    title="Qafza Olist Delivery Delay API",
    version=config["project"]["version"],
)


@app.get(
    "/health",
    response_model=HealthResponse,
)
def health():
    return {
        "status": "ok"
    }


@app.get(
    "/model",
    response_model=ModelInfoResponse,
)
def model_info():
    return {
        "model_name": config["mlflow"][
            "registered_model_

In [74]:
from pathlib import Path

path = Path("src/logging_config.py")
print(path.read_text(encoding="utf-8"))

from pathlib import Path
import logging


def configure_logging(
    log_directory: str,
    log_filename: str,
    log_level: str,
) -> None:
    log_dir = Path(log_directory)
    log_dir.mkdir(parents=True, exist_ok=True)

    log_file = log_dir / log_filename

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )

    root_logger = logging.getLogger()

    numeric_level = getattr(
        logging,
        log_level.upper(),
        logging.INFO,
    )

    root_logger.setLevel(numeric_level)

    if root_logger.handlers:
        return

    console_handler = logging.StreamHandler()
    console_handler.setLevel(numeric_level)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        log_file,
        encoding="utf-8",
    )
    file_handler.setLevel(numeric_level)
    file_handler.setFormatter(formatter)

    root_logger.addHandler(console_handler)
    root_logger.addHandler(file_handler)


In [77]:
import inspect
import src.prediction

print(inspect.getsource(src.prediction))

import logging

import mlflow
import mlflow.sklearn
import pandas as pd

from src.config import load_config


logger = logging.getLogger(__name__)


def load_model():
    """
    Load the registered trained model from MLflow.

    The model is loaded only for prediction.
    It is never trained again during inference.
    """
    try:
        config = load_config()

        tracking_uri = config["mlflow"]["tracking_uri"]
        registered_model_name = config["mlflow"]["registered_model_name"]
        model_alias = config["mlflow"]["model_alias"]

        mlflow.set_tracking_uri(tracking_uri)

        model_uri = (
            f"models:/{registered_model_name}@{model_alias}"
        )

        model = mlflow.sklearn.load_model(model_uri)

        logger.info(
            "Model loaded successfully from MLflow Registry: %s",
            model_uri,
        )

        return model

    except Exception:
        logger.exception(
            "Failed to load model from MLflow Registry."
   

In [78]:
import importlib
import app.main
import src.prediction

importlib.reload(src.prediction)
importlib.reload(app.main)

print("Reload completed.")

2026-09-24 23:14:22,794 | INFO | src.config | Configuration loaded successfully.


Reload completed.


In [79]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

response = client.post(
    "/predict",
    json={
        "seller_customer_distance_km": 300.0,
        "total_freight": 25.0,
        "total_price": 150.0,
        "num_items": 1,
        "num_unique_products": 1,
        "num_unique_sellers": 1,
        "total_product_weight_g": 500.0,
        "total_product_volume_cm3": 1000.0,
        "customer_state": "SP",
        "order_purchase_timestamp": "2018-01-01T10:00:00",
        "order_estimated_delivery_date": "2018-01-15T00:00:00",
    }
)

print("Status code:", response.status_code)
print("Response:", response.json())

2026-09-24 23:15:26,476 | WARNING | py.warnings | C:\Users\Ahlam\miniconda3\envs\olist\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa

2026-09-24 23:15:26,604 | INFO | great_expectations.data_context.data_context.file_data_context | FileDataContext loading fluent config
2026-09-24 23:15:26,742 | INFO | great_expectations.datasource.fluent.config | Loading 'datasources' ->
[{'assets': [...],
  'base_directory': 'C:\\Users\\Ahlam\\Qafza_Training\\data\\artifacts',
  'id': '862dc565-be2c-47d4-8768-a4a05d6fdb1a',
  'name': 'olist_artifacts',
  'type': 'pandas_filesystem'},
 {'assets': [...],
  'id': 'b9ae60be-d0e8-417c-a29c-a7190fff9a47',
  'name': 'olist_inference',
  'type': 'pandas'}]
2026-09-24 23:15:26,744 | INFO | great_expectations.datasource.fluent.fluent_base_model | FileDataAsset.dict() - missing `config_pr

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

2026-09-24 23:15:27,969 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:15:27,977 | INFO | src.preprocessing | Preprocessor loaded successfully from: data/artifacts/features/05_preprocessor.joblib
2026-09-24 23:15:27,993 | INFO | src.preprocessing | Features transformed successfully. Input shape: (1, 13), output shape: (1, 39)
2026-09-24 23:15:28,009 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:15:28,026 | INFO | src.config | Configuration loaded successfully.


2026-09-24 23:15:47,475 | INFO | src.prediction | Model loaded successfully from MLflow Registry: models:/qafza-olist-delivery-delay@production
2026-09-24 23:15:47,475 | INFO | src.prediction | Prediction completed successfully. Input rows: 1, threshold: 0.67
2026-09-24 23:15:47,487 | INFO | app.main | Prediction request | input={'seller_customer_distance_km': 300.0, 'total_freight': 25.0, 'total_price': 150.0, 'num_items': 1, 'num_unique_products': 1, 'num_unique_sellers': 1, 'total_product_weight_g': 500.0, 'total_product_volume_cm3': 1000.0, 'customer_state': 'SP', 'order_purchase_timestamp': datetime.datetime(2018, 1, 1, 10, 0), 'order_estimated_delivery_date': datetime.datetime(2018, 1, 15, 0, 0)} | output={'prediction': 0, 'probability': 0.5455231722979448} | latency=20.980127s | model_version=production
2026-09-24 23:15:47,500 | INFO | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


Status code: 200
Response: {'prediction': 0, 'probability': 0.5455231722979448, 'model_version': 'production'}


In [80]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

response = client.post(
    "/predict/batch",
    json={
        "orders": [
            {
                "seller_customer_distance_km": 300.0,
                "total_freight": 25.0,
                "total_price": 150.0,
                "num_items": 1,
                "num_unique_products": 1,
                "num_unique_sellers": 1,
                "total_product_weight_g": 500.0,
                "total_product_volume_cm3": 1000.0,
                "customer_state": "SP",
                "order_purchase_timestamp": "2018-01-01T10:00:00",
                "order_estimated_delivery_date": "2018-01-15T00:00:00",
            },
            {
                "seller_customer_distance_km": 100.0,
                "total_freight": 10.0,
                "total_price": 80.0,
                "num_items": 2,
                "num_unique_products": 2,
                "num_unique_sellers": 1,
                "total_product_weight_g": 800.0,
                "total_product_volume_cm3": 1500.0,
                "customer_state": "RJ",
                "order_purchase_timestamp": "2018-02-10T14:00:00",
                "order_estimated_delivery_date": "2018-02-25T00:00:00",
            },
        ]
    },
)

print("Status code:", response.status_code)
print("Response:", response.json())

2026-09-24 23:20:05,952 | INFO | great_expectations.data_context.data_context.file_data_context | FileDataContext loading fluent config
2026-09-24 23:20:06,045 | INFO | great_expectations.datasource.fluent.config | Loading 'datasources' ->
[{'assets': [...],
  'base_directory': 'C:\\Users\\Ahlam\\Qafza_Training\\data\\artifacts',
  'id': '862dc565-be2c-47d4-8768-a4a05d6fdb1a',
  'name': 'olist_artifacts',
  'type': 'pandas_filesystem'},
 {'assets': [...],
  'id': 'b9ae60be-d0e8-417c-a29c-a7190fff9a47',
  'name': 'olist_inference',
  'type': 'pandas'}]
2026-09-24 23:20:06,050 | INFO | great_expectations.datasource.fluent.fluent_base_model | FileDataAsset.dict() - missing `config_provider`, skipping config substitution
2026-09-24 23:20:06,054 | INFO | great_expectations.datasource.fluent.fluent_base_model | _PandasDataAsset.dict() - missing `config_provider`, skipping config substitution
2026-09-24 23:20:06,252 | INFO | great_expectations.data_context.data_context.abstract_data_context |

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

2026-09-24 23:20:06,710 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:20:06,737 | INFO | src.preprocessing | Preprocessor loaded successfully from: data/artifacts/features/05_preprocessor.joblib
2026-09-24 23:20:06,756 | INFO | src.preprocessing | Features transformed successfully. Input shape: (2, 13), output shape: (2, 39)
2026-09-24 23:20:06,770 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:20:06,780 | INFO | src.config | Configuration loaded successfully.


2026-09-24 23:20:11,468 | INFO | src.prediction | Model loaded successfully from MLflow Registry: models:/qafza-olist-delivery-delay@production
2026-09-24 23:20:11,476 | INFO | src.prediction | Prediction completed successfully. Input rows: 2, threshold: 0.67
2026-09-24 23:20:11,477 | INFO | app.main | Batch prediction request | input=[{'seller_customer_distance_km': 300.0, 'total_freight': 25.0, 'total_price': 150.0, 'num_items': 1, 'num_unique_products': 1, 'num_unique_sellers': 1, 'total_product_weight_g': 500.0, 'total_product_volume_cm3': 1000.0, 'customer_state': 'SP', 'order_purchase_timestamp': datetime.datetime(2018, 1, 1, 10, 0), 'order_estimated_delivery_date': datetime.datetime(2018, 1, 15, 0, 0)}, {'seller_customer_distance_km': 100.0, 'total_freight': 10.0, 'total_price': 80.0, 'num_items': 2, 'num_unique_products': 2, 'num_unique_sellers': 1, 'total_product_weight_g': 800.0, 'total_product_volume_cm3': 1500.0, 'customer_state': 'RJ', 'order_purchase_timestamp': datetime.

Status code: 200
Response: {'predictions': [{'prediction': 0, 'probability': 0.5455231722979448, 'model_version': 'production'}, {'prediction': 1, 'probability': 0.7490385278209573, 'model_version': 'production'}]}


In [81]:
from pathlib import Path

log_file = Path("data/artifacts/logs/inference.log")
print("\n".join(log_file.read_text(encoding="utf-8").splitlines()[-8:]))

2026-09-24 23:20:06,737 | INFO | src.preprocessing | Preprocessor loaded successfully from: data/artifacts/features/05_preprocessor.joblib
2026-09-24 23:20:06,756 | INFO | src.preprocessing | Features transformed successfully. Input shape: (2, 13), output shape: (2, 39)
2026-09-24 23:20:06,770 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:20:06,780 | INFO | src.config | Configuration loaded successfully.
2026-09-24 23:20:11,468 | INFO | src.prediction | Model loaded successfully from MLflow Registry: models:/qafza-olist-delivery-delay@production
2026-09-24 23:20:11,476 | INFO | src.prediction | Prediction completed successfully. Input rows: 2, threshold: 0.67
2026-09-24 23:20:11,477 | INFO | app.main | Batch prediction request | input=[{'seller_customer_distance_km': 300.0, 'total_freight': 25.0, 'total_price': 150.0, 'num_items': 1, 'num_unique_products': 1, 'num_unique_sellers': 1, 'total_product_weight_g': 500.0, 'total_product_volume_cm3': 1000.0, 'customer